In [216]:
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [217]:
from dotenv import load_dotenv
import os
from groq import Groq

load_dotenv()

True

In [218]:
import os

from groq import Groq

client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)


Fast language models are crucial in various applications and have numerous benefits. Here are some key reasons highlighting their importance:

1. **Efficient Processing**: Fast language models can process and analyze vast amounts of data quickly, which is essential for applications that require real-time or near-real-time processing, such as chatbots, virtual assistants, and text classification systems.
2. **Improved User Experience**: Faster language models enable responsive and interactive systems, which lead to a better user experience. For example, fast language models can respond to user queries quickly, making conversational AI systems more engaging and effective.
3. **Scalability**: Fast language models can handle a large volume of requests and data, making them ideal for large-scale applications, such as language translation, text summarization, and sentiment analysis.
4. **Reduced Latency**: Fast language models minimize latency, which is critical in applications where time is

In [219]:
class Agent:
    def __init__(self, client: Groq, system: str = "") -> None:
        self.client = client
        self.system = system
        self.messages: list = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message=""):
        if message:
            self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile", messages=self.messages
        )
        return completion.choices[0].message.content

In [220]:
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

calculate:
e.g. calculate: 4 * 7 / 3
Runs a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary

get_planet_mass:
e.g. get_planet_mass: Earth
returns weight of the planet in kg

Example session:

Question: What is the mass of Earth times 2?
Thought: I need to find the mass of Earth
Action: get_planet_mass: Earth
PAUSE 

You will be called again with this:

Observation: 5.972e24

Thought: I need to multiply this by 2
Action: calculate: 5.972e24 * 2
PAUSE

You will be called again with this: 

Observation: 1,1944×10e25

If you have the answer, output it as the Answer.

Answer: The mass of Earth times 2 is 1,1944×10e25.

Now it's your turn:
""".strip()

In [221]:
# ferramenta = são as funções que o agente pode chamar
def calculate(operation: str) -> float:
    return eval(operation)


def get_planet_mass(planet) -> float: # Nessa função eu poderia usar uma API para pegar a massa dos planetas, mas como é um exemplo, vou usar valores fixos
    match planet.lower():
        case "earth":
            return 5.972e24
        case "jupiter":
            return 1.898e27
        case "mars":
            return 6.39e23
        case "mercury":
            return 3.285e23
        case "neptune":
            return 1.024e26
        case "saturn":
            return 5.683e26
        case "uranus":
            return 8.681e25
        case "venus":
            return 4.867e24
        case _:
            return 0.0

In [222]:
uncle_sam = Agent(client=client, system=system_prompt)

In [223]:
result = uncle_sam("What is the mass of Earth times 5?")
print(result)

Thought: I need to find the mass of Earth and then multiply it by 5 to get the result.
Action: get_planet_mass: Earth
PAUSE


In [224]:
uncle_sam.messages # Olha o histórico de mensagens do agente

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944×10e25\n\nIf you have the

In [225]:
result = uncle_sam()
print(result)

In [226]:
observation = get_planet_mass("Earth")
print(observation)

5.972e+24


In [227]:
next_promt = f"Observation: {observation}"
result = uncle_sam(next_promt)
print(result)

Thought: Now that I have the mass of Earth, I need to multiply this by 5 to get the result.
Action: calculate: 5.972e24 * 5
PAUSE


In [228]:
uncle_sam.messages # Olha o histórico de mensagens do agente

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944×10e25\n\nIf you have the

In [229]:
result = uncle_sam()
print(result)
# O agente pode continuar a conversa, chamando o modelo novamente

In [230]:
observation = calculate("3.285e23 * 5")
print(observation)

1.6425e+24


In [231]:
next_promt = f"Observation: {observation}"
result = uncle_sam(next_promt)
print(result)

Thought: The calculation is complete, and I have the result of the mass of Earth times 5.
Answer: The mass of Earth times 5 is 2.986e+25


In [232]:
uncle_sam.messages # Olha o histórico de mensagens do agente

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\ncalculate:\ne.g. calculate: 4 * 7 / 3\nRuns a calculation and returns the number - uses Python so be sure to use floating point syntax if necessary\n\nget_planet_mass:\ne.g. get_planet_mass: Earth\nreturns weight of the planet in kg\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE \n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply this by 2\nAction: calculate: 5.972e24 * 2\nPAUSE\n\nYou will be called again with this: \n\nObservation: 1,1944×10e25\n\nIf you have the

In [ ]:
import re # Regex para extrair a ação e o argumento

def agent_loop(max_iterations, query):
    agent = Agent(client, system_prompt)
    tools = ['calculate', 'get_planet_mass']
    next_prompt = query
    i = 0
    while i < max_iterations:
        i+= 1
        result = agent(next_prompt)
        print(result)
        
        if "PAUSE" in result and "Action" in result:
            action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
            chosen_tool = action[0][0]
            arg = action[0][1]

            if chosen_tool in tools:
                result_tool = eval(f"{chosen_tool}('{arg}')")
                next_prompt = f"Observation: {result_tool}"

            else:
                next_prompt = "Observation: Tool not found"

            print(next_prompt)
            continue

        if "Answer" in result:
            break

agent_loop(max_iterations=10, query="What is the mass of Earth plus the mass of Saturn and all of that times 2?")

TypeError: agent_loop() missing 1 required positional argument: 'max_iterations'